## 회원 CRUD - auth.users, user_details
- 삭제 : deleted_at에 날짜와 시간이 업데이트되면 소프트 삭제
- 조회시에는 deleted_at IS NOT NULL로 조회될 수 있도록 한다.

In [ ]:
from supabase_client import get_supabase_client

supabase = get_supabase_client()

In [1]:
class user_service:
    pass

import datetime
from zoneinfo import ZoneInfo

class user_service():
    def __init__(self, supabase):
        user = supabase.auth.get_user()
        if not user:
            raise PermissionError("로그인이 필요합니다.")

        print("정상 로그인 되었습니다.")
        self.supabase = supabase
        self.user = user.user

    def get_users(self):
        response = (
            self.supabase.table("user_details")
                .select("*")
                .is_("deleted_at", "null")
                .execute()
        )
        return response.data


    def get_user_info(self):
        response = (
            self.supabase.table("user_details")
                .select("id, type, cellphone, zipcode, address, address_sub")
                .eq("id", self.user.id)
                .execute()
        )

        return response.data

    def add_user(self, type, cellphone, zipcode, address, address_sub):
        response = (
            self.supabase.table("user_details")
                .insert({
                    "id" : self.user.id,
                    "type" : type,
                    "cellphone" : cellphone,
                    "zipcode" : zipcode,
                    "address" : address,
                    "address_sub" : address_sub
                })
                .execute()
        )

        return response.data

    def set_user(self, cellphone, zipcode, address, address_sub):
        if not(cellphone or zipcode or address or address_sub): return

        up_data = {}
        if cellphone: up_data['cellphone'] = cellphone
        if zipcode: up_data['zipcode'] = zipcode
        if address: up_data['address'] = address
        if address_sub: up_data['address_sub'] = address_sub
        up_data['modified_at'] = datetime.datetime.now(ZoneInfo("Asia/Seoul"))

        response = (
            self.supabase.table("user_details")
                .update(up_data)
                .eq("id", self.user.id)
                .execute()
        )

        return response.data

    def remove_user(self):
        response = (
            self.supabase.table("user_details")
                .update({
                    "deleted_at" : datetime.datetime.now(ZoneInfo("Asia/Seoul"))
                })
                .eq("id", self.user.id)
                .execute()
        )

        return response.data
